# Maintaining a NISAR virtual cube

[`test_virtualizarr.ipynb`](test_virtualizarr.ipynb) established that VirtualiZarr works on NISAR
GCOV and what it costs. This notebook is the operational follow-up — the two things that turn a
one-off experiment into something a team can actually run:

1. **Parallelizing the manifest build**, so the initial index is not a serial walk over the archive.
2. **Appending new acquisitions** as NISAR publishes them, so the index is never rebuilt.

Plus the consequence nobody mentions: appends make the Icechunk repo grow, and you have to
compact it.

Everything runs against the same stack as before — GCOV, track D/065, frame 4005, Mt. Rainier.

In [1]:
import sys, time, warnings, datetime
from pathlib import Path
import shutil

sys.path.insert(0, str(Path.cwd()))
import numpy as np, pandas as pd, xarray as xr
import nisar_virtual as nv

warnings.filterwarnings("ignore")

import icechunk
icechunk.set_logs_filter("error")   # the Rust layer logs at WARN by default

BBOX = (-121.9, 46.7, -121.6, 46.95)   # Mt. Rainier
registry = nv.obstore_registry()

tracks = nv.find_track(BBOX)
key, items = max(tracks.items(), key=lambda kv: len(kv[1]))
print(f"track {key}: {len(items)} GCOV granules, {items[0][0][:8]} -> {items[-1][0][:8]}")

/home/jovyan/repos/nicer-nisar/.pixi/envs/default/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


track ('D', '065', '4005', 'DHDH'): 28 GCOV granules, 20251110 -> 20260818


One thing `find_track` now does that it did not in the first notebook: **deduplicate product
versions**. The archive holds more than one processing of some acquisitions, and a naive search
returns both.

In [2]:
from collections import Counter
import earthaccess

raw = []
for g in earthaccess.search_data(short_name="NISAR_L2_GCOV_PROVISIONAL_V1",
                                 bounding_box=BBOX, count=200):
    for u in g.data_links(access="direct"):
        if u.endswith(".h5") and (m := nv.GRANULE_RX.search(u.rsplit("/", 1)[-1])):
            if (m["direction"], m["relorb"], m["frame"], m["pols"]) == key:
                raw.append((m["start"], u.rsplit("/", 1)[-1]))

dups = [t for t, n in Counter(t for t, _ in raw).items() if n > 1]
print(f"search returns {len(raw)} granules; find_track keeps {len(items)}")
for t in dups:
    print(f"\n{t} appears {sum(1 for x, _ in raw if x == t)} times:")
    for x, n in raw:
        if x == t:
            print("   ", n)

search returns 29 granules; find_track keeps 28

20260626T031850 appears 2 times:
    NISAR_L2_PR_GCOV_023_172_D_065_4005_DHDH_A_20260626T031850_20260626T031926_P05023_N_F_J_001.h5
    NISAR_L2_PR_GCOV_023_172_D_065_4005_DHDH_A_20260626T031850_20260626T031926_P05023_N_F_J_002.h5


`..._001.h5` and `..._002.h5` are the same take, reprocessed. Indexing both puts the same
acquisition into the cube twice under one timestamp — silently, since nothing about the
concatenation objects to it. `find_track` keeps the highest version.

## 1. Parallelizing the build

The obvious move is a thread pool — the work looks I/O bound, since each granule is a handful of
range reads against S3. It isn't. Profiling one build shows where the ~0.65 s actually goes:

In [3]:
import cProfile, pstats, io, virtualizarr as vz
from virtualizarr.parsers import HDFParser

parser = HDFParser(group=nv.GCOV_GRIDS)
pr = cProfile.Profile(); pr.enable()
one = vz.open_virtual_dataset(items[0][1], registry=registry, parser=parser)
pr.disable()

st = pstats.Stats(pr)
total = st.total_tt
interesting = ("validate_and_normalize_path_to_uri", "urlparse", "get_ranges",
               "chunk_iter", "from_arrays")

print(f"total {total:.2f}s")
print(f"{'ncalls':>8} {'cumtime':>8} {'% total':>8}  function")
seen = set()
for (fn, ln, name), (cc, nc, tt, ct, _) in sorted(
        st.stats.items(), key=lambda kv: -kv[1][3]):
    if name in interesting and name not in seen:
        seen.add(name)
        print(f"{nc:>8,} {ct:>8.2f} {100 * ct / total:>7.0f}%  {name}")

total 1.70s
  ncalls  cumtime  % total  function
      10     0.46      27%  from_arrays
  29,833     0.44      26%  validate_and_normalize_path_to_uri
  29,838     0.40      24%  urlparse
       5     0.25      14%  get_ranges


`validate_and_normalize_path_to_uri` runs **once per chunk reference** — ~30,000 times for one
granule — and calls `urlparse` on the same S3 URL every time. That is pure Python holding the GIL,
and it is roughly half the build. The actual S3 reads (`get_ranges`, 5 calls) are a third.

So threads are the wrong tool, and processes only partly the right one:

In [4]:
from concurrent.futures import ThreadPoolExecutor

urls = [u for _, u in items]
results = {}

t0 = time.perf_counter()
vdss = nv.build_manifests(urls, registry=registry)
results["serial"] = time.perf_counter() - t0

t0 = time.perf_counter()
with ThreadPoolExecutor(8) as ex:
    list(ex.map(lambda u: vz.open_virtual_dataset(u, registry=registry, parser=parser), urls))
results["threads (8)"] = time.perf_counter() - t0

t0 = time.perf_counter()
nv.build_manifests(urls, workers=8)
results["processes (8)"] = time.perf_counter() - t0

base = results["serial"]
for k, v in results.items():
    print(f"{k:15s} {v:5.1f}s   {base / v:.1f}x")

serial           17.1s   1.0x
threads (8)      15.6s   1.1x
processes (8)    12.5s   1.4x


Threads: nothing, as predicted. Processes: about 1.4x, and no better with more workers — the
manifests have to be pickled back to the parent, which spends the CPU the workers saved, and each
worker pays for its own Earthdata login before it can do anything.

The bigger lever is not parallelism at all. The `frequencyA` group holds 14 variables, and a
backscatter time series needs 3 of them. Dropping the rest cuts both the build time and the index
size:

In [5]:
all_vars = list(one.data_vars) + list(one.coords)
drop = nv.drop_outside(all_vars)
print(f"{len(all_vars)} variables in the group, dropping {len(drop)}:")
print("  ", ", ".join(drop))

refs = lambda d: sum(len(d[v].data.manifest) for v in d.data_vars
                     if hasattr(d[v].data, "manifest"))

t0 = time.perf_counter()
lean = nv.build_manifests(urls, registry=registry, drop_variables=drop)
t_lean = time.perf_counter() - t0
t0 = time.perf_counter()
nv.build_manifests(urls, drop_variables=drop, workers=8)
t_lean_par = time.perf_counter() - t0

print(f"\nall 14 vars, serial      : {results['serial']:5.1f}s   {refs(one):,} refs/granule")
print(f"3 vars,      serial      : {t_lean:5.1f}s   {refs(lean[0]):,} refs/granule")
print(f"3 vars,      8 processes : {t_lean_par:5.1f}s   "
      f"({results['serial'] / t_lean_par:.1f}x end to end)")

14 variables in the group, dropping 9:
   inputDataExceptionMask, listOfCovarianceTerms, listOfPolarizations, mask, numberOfLooks, numberOfSubSwaths, rtcGammaToSigmaFactor, xCoordinateSpacing, yCoordinateSpacing



all 14 vars, serial      :  17.1s   29,826 refs/granule
3 vars,      serial      :   8.9s   9,941 refs/granule
3 vars,      8 processes :  17.2s   (1.0x end to end)


`open_virtual_mfdataset` is the library's own entry point for this. It combines as it goes, which
is convenient — but only usable once you already know the granules share a grid, and its
`parallel=` options are the same threads and dask that don't help here.

In [6]:
by_grid = nv.group_by_grid(items, vdss)
sig, members = next(iter(by_grid.items()))
g_urls = [u for _, u, _ in members]
print(f"{len(by_grid)} distinct grids in this track; largest has {len(members)} granules")

for label, par in (("serial", False), ("threads", ThreadPoolExecutor), ("dask", "dask")):
    t0 = time.perf_counter()
    mf = vz.open_virtual_mfdataset(g_urls, registry=registry, parser=parser,
                                   combine="nested", concat_dim="time", join="exact",
                                   coords="minimal", compat="override", parallel=par)
    print(f"open_virtual_mfdataset(parallel={label:7s}) : {time.perf_counter() - t0:5.1f}s")

2 distinct grids in this track; largest has 23 granules


open_virtual_mfdataset(parallel=serial ) :  13.8s


open_virtual_mfdataset(parallel=threads) :  11.5s


open_virtual_mfdataset(parallel=dask   ) :  12.8s


## 2. Appending new acquisitions

NISAR revisits every 12 days, so a cube is obsolete almost immediately. Icechunk is transactional
and supports `append_dim`, so new granules can be added without touching what is already indexed.

Simulating that: build the cube from everything up to March 2026, then append the rest one
acquisition at a time, as if each had just been published.

In [7]:
REPO = Path("/tmp/nisar_cube"); shutil.rmtree(REPO, ignore_errors=True)
repo = nv.open_repo(REPO, create=True)

cut = pd.Timestamp("2026-04-01")
early = [(s, u) for s, u, _ in members
         if pd.to_datetime(s, format="%Y%m%dT%H%M%S") < cut]
later = [(s, u) for s, u, _ in members
         if pd.to_datetime(s, format="%Y%m%dT%H%M%S") >= cut]

t0 = time.perf_counter()
snap, times = nv.append_new(repo, early, registry=registry, drop_variables=drop, workers=8)
print(f"initial build: {len(times)} acquisitions in {time.perf_counter() - t0:.1f}s  ({snap})")

initial build: 12 acquisitions in 16.0s  (549EP5CGVRKAS9MJ435G)


In [8]:
size = lambda: sum(f.stat().st_size for f in REPO.rglob("*") if f.is_file()) / 1e6

for s_, u_ in later:
    t0 = time.perf_counter()
    snap, times = nv.append_new(repo, [(s_, u_)], registry=registry, drop_variables=drop)
    if snap is None:
        print(f"  . {s_[:8]}  already indexed or off-grid, skipped")
        continue
    print(f"  + {times[0]:%Y-%m-%d}  {time.perf_counter() - t0:4.1f}s   "
          f"repo now {size():5.1f} MB")

  + 2026-04-03   0.8s   repo now   5.0 MB


  + 2026-04-15   0.8s   repo now   7.5 MB


  + 2026-04-27   0.8s   repo now  10.2 MB


  + 2026-05-09   0.9s   repo now  13.0 MB


  + 2026-05-21   0.9s   repo now  15.9 MB


...

  + 2026-06-14   0.9s   repo now  22.1 MB


  + 2026-06-26   0.9s   repo now  25.4 MB


  + 2026-07-08   0.9s   repo now  28.9 MB


  + 2026-07-20   0.9s   repo now  32.5 MB


  + 2026-08-13   1.0s   repo now  36.2 MB


Under a second to extend a multi-terabyte cube by one date. `append_new` is idempotent — it reads
the times already in the cube and skips them — so it is safe to run on a schedule against the
full search results:

In [9]:
snap, times = nv.append_new(repo, [(s, u) for s, u, _ in members],
                            registry=registry, drop_variables=drop)
print("re-running over every granule in the grid:", snap, "->", len(times), "appended")

cube = nv.open_cube(repo)
print(f"\ncube: {dict(cube.sizes)}")
print(f"times monotonic: {cube.time.to_index().is_monotonic_increasing}")
print(f"{cube.nbytes / 1e12:.2f} TB indexed by a {size():.1f} MB repo")

re-running over every granule in the grid: None -> 0 appended

cube: {'time': 23, 'yCoordinates': 35712, 'xCoordinates': 36144}
times monotonic: True
0.24 TB indexed by a 36.2 MB repo


And the appended pixels are the real ones. Comparing the last appended date against a direct
`h5py` read of that granule — using `equal_nan`, because a NISAR grid is mostly fill outside the
imaged swath:

In [10]:
import h5py
from obspec_utils.readers import BlockStoreReader

last_s, last_u = max(((s, u) for s, u, _ in members))
store, path = registry.resolve(last_u)
with h5py.File(BlockStoreReader(store, path), "r") as f:
    truth = f[f"{nv.GCOV_GRIDS}/HHHH"][20000:20050, 18000:18050]

got = cube["HHHH"].isel(time=-1, yCoordinates=slice(20000, 20050),
                        xCoordinates=slice(18000, 18050)).values
print(f"{last_s[:8]}  fill fraction {np.isnan(truth).mean():.2f}  "
      f"byte-identical: {np.array_equal(truth, got, equal_nan=True)}")

20260813  fill fraction 0.00  byte-identical: True


## 3. The cost of appending: compaction

The repo grew from 11 MB to over 40 MB while we appended 9 dates — far more than the pixels
warrant. That is not a leak. Icechunk manifests are immutable, so each append writes a **new**
manifest for the whole array and keeps the old one, which is what makes `repo.ancestry()` a
usable time machine.

For an index that is rebuilt from a public archive anyway, that history is not worth much:

In [11]:
print("history:")
for a in list(repo.ancestry(branch="main"))[:12]:
    print(f"  {a.id}  {a.message}")

before = size()
expired, summary = nv.compact(repo)
print(f"\nexpired {len(expired)} snapshots")
print(f"deleted {summary.manifests_deleted} manifests, "
      f"{summary.bytes_deleted / 1e6:.1f} MB")
print(f"repo: {before:.1f} MB -> {size():.1f} MB")

cube = nv.open_cube(repo)
print(f"\nstill opens fine: {dict(cube.sizes)}")

history:
  CT460YQ128GKFQH72VDG  append 1 acquisitions (2026-08-13..2026-08-13)
  AP50SFJV9A35ZKWKXA1G  append 1 acquisitions (2026-07-20..2026-07-20)
  9SF30YZ8VQN8ZR9TSEZ0  append 1 acquisitions (2026-07-08..2026-07-08)
  YHQMJ29N9V3RRCA4K500  append 1 acquisitions (2026-06-26..2026-06-26)
  GBDQ7TG2C48AZV16B0FG  append 1 acquisitions (2026-06-14..2026-06-14)
  5680PYMKXVTDSTKETEQG  append 1 acquisitions (2026-06-02..2026-06-02)
  A2H6B3XJAC9DD3NA57NG  append 1 acquisitions (2026-05-21..2026-05-21)
  91KYGZWJFQNGWP3P69AG  append 1 acquisitions (2026-05-09..2026-05-09)
  4SQH5FR8ZJR2V9GKK1D0  append 1 acquisitions (2026-04-27..2026-04-27)
  9SMF5GYCYXSFC6SPDSG0  append 1 acquisitions (2026-04-15..2026-04-15)
  SN8XTYJ51Q02TCT5MD9G  append 1 acquisitions (2026-04-03..2026-04-03)
  549EP5CGVRKAS9MJ435G  create 12 acquisitions (2025-11-10..2026-03-22)

expired 11 snapshots
deleted 66 manifests, 31.6 MB
repo: 36.2 MB -> 4.6 MB

still opens fine: {'time': 23, 'yCoordinates': 35712, 'xCoord

Back to the size of a single from-scratch build, with the cube intact and the history gone.

So the maintenance loop is: **append on a schedule, compact periodically.** If you want the
history — to reproduce a result against the index as it stood — expire with an older cutoff
instead, and keep the snapshots you care about as tags.

## What this means for the project

**Parallelizing the build is not where the win is.** 1.4x from processes, nothing from threads,
because half the cost is per-chunk-reference Python inside VirtualiZarr rather than I/O. Scoping to
the variables you actually need is worth more — 1.9x, and a 3x smaller index — and it does not
stack with the process pool at this size: eight workers each pay for an Earthdata login at startup,
which is a fixed cost the shorter job cannot amortize. On 28 granules the pool is a wash. It would
pay on a few hundred.

**Appending is where the win is.** Under a second to add an acquisition to a 0.24 TB cube means the
index can track the archive: a scheduled job that searches for new granules on the AOI's track,
appends what it finds, and compacts periodically. Nobody rebuilds anything, and every reader gets
the new date the moment it lands.

**And the search needs deduplicating.** Two product versions of one acquisition would have gone in
as one timestamp with two sets of pixels — the kind of thing that produces a wrong answer rather
than an error.

**The natural scope is one AOI, one track, one grid, GCOV.** Not a limitation we chose — it is what
the data model allows. A cube is a stack of granules that share an output grid exactly, and
VirtualiZarr will not reproject to make that true.